# 01. MAF 用ツールを作成する (カスタムツール)

## コードの解説

MAF (Microsoft Agent Framework) の **カスタムツール** は、通常の Python 関数として定義します。
エージェントはユーザーの質問を分析し、必要なツールを自律的に選択・実行して応答します。

### ツール定義の 3 つのポイント

| ポイント | 内容 |
|---------|------|
| **`docstring`** | 関数の説明。エージェントがどのツールを使うか判断する際に参照する |
| **`Annotated` + `Field`** | 引数の型と説明を定義。エージェントがパラメータを正しく設定するために使う |
| **戻り値の型** | 関数が返す値の型（`str`, `dict` など）を明示する |

### ツール呼び出しの流れ

```
ユーザー質問
    ↓
エージェントがツール選択を判断
    ↓
ツール関数を実行（引数を自動設定）
    ↓
結果をもとにエージェントが応答を生成
```

このセルでは 2 つのカスタムツール（天気取得・通貨換算）を定義し、単体テストを実行します。

In [ ]:
from typing import Annotated
from random import randint, choice
from pydantic import Field


# ---------------------------------------------------------------
# カスタムツール 1: 天気取得
# ポイント1: docstring にツールの目的を明確に書く（エージェントが読む）
# ポイント2: 引数を Annotated[型, Field(description=...)] で定義する
# ポイント3: 戻り値の型を str など明示する
# ---------------------------------------------------------------
def get_weather(
    location: Annotated[
        str,
        Field(description="天気を取得したい都市名（例: 'Tokyo', 'Paris', 'New York'）"),
    ],
) -> str:
    """指定した場所の現在の天気を取得します。天候の状態と気温（摂氏）を返します。"""
    conditions = ["晴れ", "晴れ時々くもり", "くもり", "雨", "嵐"]
    temp = randint(10, 35)
    return f"{location}の天気: {choice(conditions)}, {temp}°C"


# ---------------------------------------------------------------
# カスタムツール 2: 通貨換算
# 複数引数がある場合も同じパターンで定義できる
# ---------------------------------------------------------------
def convert_currency(
    amount: Annotated[float, Field(description="換算したい金額")],
    from_currency: Annotated[str, Field(description="元の通貨コード（例: 'USD', 'EUR', 'JPY'）")],
    to_currency: Annotated[str, Field(description="換算先の通貨コード（例: 'USD', 'EUR', 'JPY'）")],
) -> str:
    """ある通貨の金額を別の通貨に換算します。概算の為替レートを使用します。"""
    rates = {"USD": 1.0, "EUR": 0.92, "GBP": 0.79, "JPY": 149.50, "AUD": 1.53}
    from_r = rates.get(from_currency.upper(), 1.0)
    to_r = rates.get(to_currency.upper(), 1.0)
    converted = (amount / from_r) * to_r
    return f"{amount} {from_currency} = {converted:.2f} {to_currency}"


# 動作確認（ツール単体テスト）
print("=== ツール単体テスト ===")
print(get_weather("Tokyo"))
print(get_weather("Paris"))
print(convert_currency(100.0, "USD", "JPY"))
print(convert_currency(5000.0, "JPY", "EUR"))